# Day 4: Country context and adoption intentions

Two research questions today.

**Question 3:** Day 3 showed that country is the strongest predictor of purchasing. But a country name explains nothing. *Which national conditions* go with each pathway: income, the electricity price, or how green the national grid already is?

**Question 4:** Among the firms that use no renewable energy, which ones plan to start within two years, and what distinguishes them from those with no plans?

**How to run it:** *Runtime > Run all*. When asked, upload **two** files together: `sme_energy_clean.csv` and `country_indicators.csv` (hold Ctrl to select both). It takes about two minutes.

## 0. Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)
SEED = 42

NEEDED = ["sme_energy_clean.csv", "country_indicators.csv"]
if not all(os.path.exists(f) for f in NEEDED):
    try:
        from google.colab import files
        print("Choose BOTH files:", NEEDED)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} in the same folder as this notebook.")

df = pd.read_csv("sme_energy_clean.csv")
indicators = pd.read_csv("country_indicators.csv").set_index("country")
df["log_employees"] = np.log1p(df["employees"])
df["log_firm_age"] = np.log1p(df["firm_age_years"].clip(lower=0))
print("Firms:", len(df), "| Countries with indicators:", len(indicators))
indicators.head()

# Part A. Question 3: which national conditions go with each pathway?

## A1. The country indicators

| Column | Meaning | Source |
|---|---|---|
| `gdp_per_capita_pps_2023_eu27_100` | Income per person, adjusted for price levels. EU average = 100 | Eurostat tec00114 |
| `electricity_price_business_2023S2_eur_kwh` | What a small business pays for electricity, second half of 2023 | Eurostat nrg_pc_205, band IB |
| `renewable_share_electricity_2023_pct` | Share of the country's electricity that comes from renewables | Eurostat nrg_ind_ren |

All refer to 2023, the year before the survey. Any further column you add to `country_indicators.csv` (for example a solar resource figure) is picked up automatically.

## A2. Two measures of each country's position

- **Raw share:** the percentage of the country's SMEs on each pathway (weighted). Simple and model-free.
- **Adjusted share:** the same thing after accounting for the mix of firms in the country. We fit a logistic regression on firm characteristics plus country and read off the country's contribution. This guards against a country looking "high" only because it has many large firms.

In [ ]:
PATHWAYS = ["No renewable", "Purchased", "On-site"]

# Raw weighted share by country
w = df["weight_country"]
raw = (w.groupby([df["country"], df["pathway"]]).sum().unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0)) * 100)
raw = raw[["Purchased", "On-site"]].round(1)

# Adjusted: country coefficients from a multinomial logistic regression that also contains firm characteristics
FIRM = ["nace_section", "size", "log_employees", "log_firm_age", "age_band", "turnover_band",
        "turnover_trend", "employment_trend", "hiring_difficulty", "sells"]
NUM_WITH_GAPS = ["log_employees", "log_firm_age"]

def make_pre(columns, frame):
    cat = [c for c in columns if not pd.api.types.is_numeric_dtype(frame[c])]
    gaps = [c for c in columns if c in NUM_WITH_GAPS]
    num = [c for c in columns if pd.api.types.is_numeric_dtype(frame[c]) and c not in NUM_WITH_GAPS]
    steps = [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
             ("gaps", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)),
                                ("scale", StandardScaler())]), gaps)]
    if num: steps.append(("num", StandardScaler(), num))
    return ColumnTransformer(steps)

y = df["pathway_code"]
pre = make_pre(FIRM + ["country"], df)
X = pre.fit_transform(df[FIRM + ["country"]])
names = list(pre.get_feature_names_out())
logit = LogisticRegression(max_iter=3000, C=1.0).fit(X, y)
country_cols = [i for i, n in enumerate(names) if n.startswith("cat__country_")]
coef = pd.DataFrame(logit.coef_[:, country_cols].T, index=[names[i].replace("cat__country_", "") for i in country_cols],
                    columns=PATHWAYS)
# Express each pathway relative to "No renewable", so a positive number means "more likely than not using renewables"
adjusted = coef[["Purchased", "On-site"]].sub(coef["No renewable"], axis=0).round(3)
adjusted.columns = ["Purchased (adjusted)", "On-site (adjusted)"]

country = raw.join(adjusted).join(indicators)
print("Do the raw and adjusted measures agree?  Purchased rho = %.2f   On-site rho = %.2f" % (
    spearmanr(country["Purchased"], country["Purchased (adjusted)"])[0],
    spearmanr(country["On-site"], country["On-site (adjusted)"])[0]))
country.sort_values("Purchased", ascending=False)

## A3. Which indicators go with which pathway?

Spearman rank correlation across the 20 countries. It runs from −1 to +1 and is not thrown off by one or two unusually high values.

**Read this with care.** There are only 20 countries, so:

- a correlation needs to be larger than about 0.45 in size to be distinguishable from chance at the usual 5% level;
- we are testing 12 correlations at once, so a "significant" result could still be luck. The **Holm correction** adjusts each p-value for the number of tests: the smallest is multiplied by 12, the next by 11, and so on. Only results that stay below 0.05 after that are solid;
- a correlation between countries does not show that the indicator causes the behaviour, or that the same relationship holds for individual firms.

In [ ]:
rows = []
for ind in indicators.columns:
    for outcome in ["Purchased", "On-site", "Purchased (adjusted)", "On-site (adjusted)"]:
        rho, p = spearmanr(country[outcome], country[ind])
        rows.append({"Indicator": ind, "Outcome": outcome, "rho": round(rho, 2), "p-value": round(p, 3)})
corr = pd.DataFrame(rows)
corr_wide = corr.pivot(index="Indicator", columns="Outcome", values="rho")[
    ["Purchased", "Purchased (adjusted)", "On-site", "On-site (adjusted)"]]
p_wide = corr.pivot(index="Indicator", columns="Outcome", values="p-value")[corr_wide.columns]
print("Spearman rho"); display(corr_wide)
print("p-values (not adjusted)"); display(p_wide)

# Holm correction across all the correlations, and a check that no single country drives a result
holm = pd.DataFrame([{"Indicator": ind, "Outcome": out, "rho": spearmanr(country[out], country[ind])[0], "p": spearmanr(country[out], country[ind])[1]}
                     for ind in indicators.columns for out in corr_wide.columns]).sort_values("p").reset_index(drop=True)
m = len(holm)
holm["Holm-adjusted p"] = np.maximum.accumulate(np.minimum(1, (m - np.arange(m)) * holm["p"].values))
holm["Still below 0.05"] = np.where(holm["Holm-adjusted p"] < 0.05, "yes", "no")
def leave_one_out(out, ind):
    v = pd.Series({c: spearmanr(country.drop(c)[out], country.drop(c)[ind])[0] for c in country.index})
    return v.min(), v.idxmin(), v.max(), v.idxmax()
holm[["Lowest rho leaving one country out", "(without)", "Highest rho leaving one country out", "(without) "]] = [leave_one_out(o, i) for o, i in zip(holm["Outcome"], holm["Indicator"])]
print("Critical value for n = %d at the 5%% level: about %.2f" % (len(country), {20: 0.447, 27: 0.382}.get(len(country), np.nan)))
holm.round(3)

### Figure 7: country indicators and the two pathways

In [ ]:
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": GRID, "text.color": INK,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK})
NICE = {"gdp_per_capita_pps_2023_eu27_100": "Income per person",
        "electricity_price_business_2023S2_eur_kwh": "Business electricity price",
        "renewable_share_electricity_2023_pct": "Renewable share of electricity"}

order = list(corr_wide["Purchased"].sort_values().index)
ypos = np.arange(len(order))
fig, ax = plt.subplots(figsize=(7.6, 1.0 + 0.9 * len(order)))
ax.barh(ypos + 0.19, corr_wide.loc[order, "Purchased"], height=0.36, color=BLUE, label="Purchased from a supplier")
ax.barh(ypos - 0.19, corr_wide.loc[order, "On-site"], height=0.36, color=ORANGE, label="Generated on-site")
for i, ind in enumerate(order):
    for off, col in [(0.19, "Purchased"), (-0.19, "On-site")]:
        v, p = corr_wide.loc[ind, col], p_wide.loc[ind, col]
        ax.text(v + (0.02 if v >= 0 else -0.02), i + off, f"{v:+.2f}" + ("*" if p < 0.05 else ""),
                va="center", ha="left" if v >= 0 else "right", fontsize=8, color=MUTED)
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_xlim(-1, 1); ax.set_yticks(ypos); ax.set_yticklabels([NICE.get(i, i) for i in order])
ax.set_xlabel(f"Rank correlation with the share of SMEs on the pathway ({len(country)} countries; * unadjusted p < 0.05)")
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.tick_params(length=0)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(-0.01, 1.0), ncol=2, fontsize=8)
ax.set_title("What goes with each pathway at country level", loc="left", fontsize=11, fontweight="bold", pad=26)
plt.tight_layout(); plt.savefig("fig7_country_indicators.png", dpi=300); plt.show()

## A4. How much of the "country" information do the indicators capture?

On Day 2, adding the country's *name* to firm characteristics raised the logistic regression's AUC by several points. If we give the model the indicators instead of the name, how much of that gain do we keep? (The regularisation setting is held at C = 0.1 for every fit here, so the scores can differ in the third decimal from Day 2.)

This uses logistic regression only. A tree model could use three numbers to work out which country a firm is in, which would defeat the purpose.

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=y, random_state=SEED)   # same split as Day 2
dfi = df.join(indicators, on="country")
IND = list(indicators.columns)

def test_auc(columns):
    pipe = Pipeline([("pre", make_pre(columns, dfi)), ("model", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1))])
    pipe.fit(dfi.loc[train_idx, columns], y.loc[train_idx])
    return roc_auc_score(y.loc[test_idx], pipe.predict_proba(dfi.loc[test_idx, columns]), multi_class="ovr")

auc_firm, auc_name = test_auc(FIRM), test_auc(FIRM + ["country"])
rows = [{"Predictors": "Firm characteristics only", "Test AUC": auc_firm},
        {"Predictors": f"Firm + country name ({df['country'].nunique()} categories)", "Test AUC": auc_name},
        {"Predictors": f"Firm + all {len(IND)} indicators", "Test AUC": test_auc(FIRM + IND)}]
rows += [{"Predictors": f"Firm + {NICE.get(i, i)} only", "Test AUC": test_auc(FIRM + [i])} for i in IND]
captured = pd.DataFrame(rows)
captured_auc = captured["Test AUC"].copy(); captured_share = 100 * (captured["Test AUC"] - auc_firm) / (auc_name - auc_firm)    # unrounded, for the results file
captured["Share of the country gain captured %"] = captured_share.round(0)
captured["Test AUC"] = captured["Test AUC"].round(3)
captured

# Part B. Question 4: who plans to adopt?

We now look only at firms that use **no** renewable energy today. The outcome is whether the firm says it plans to use renewable energy within two years (survey question Q2.3).

Same method as Days 2 and 3, on a yes/no outcome: hold back a test set, compare logistic regression with XGBoost, then explain the better model with SHAP.

In [ ]:
non = df[df["pathway"] == "No renewable"].reset_index(drop=True)
yb = non["plans_renewable"]
plan_weighted = 100 * np.average(yb, weights=non["weight_eu27"])
print(f"Non-users: {len(non)} | plan to adopt: {int(yb.sum())} ({100 * yb.mean():.1f}% unweighted, {plan_weighted:.1f}% weighted)")

BEHAVIOUR = ([c for c in df.columns if c.startswith(("act_", "sup_", "ext_", "bar_", "help_"))]
             + ["n_other_actions", "n_barriers", "asked_followups", "investment", "climate_strategy",
                "green_products", "green_jobs", "recycled_materials", "lifespan_policy"])
PREDICTORS = FIRM + ["country"] + BEHAVIOUR

tr, te = train_test_split(non.index, test_size=0.20, stratify=yb, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

pre_b = make_pre(PREDICTORS, non)
Xtr, Xte = pre_b.fit_transform(non.loc[tr, PREDICTORS]), pre_b.transform(non.loc[te, PREDICTORS])
enc_names = list(pre_b.get_feature_names_out())

lg = GridSearchCV(LogisticRegression(max_iter=3000, class_weight="balanced"), {"C": [0.01, 0.1, 1.0]},
                  scoring="roc_auc", cv=cv, n_jobs=-1).fit(Xtr, yb.loc[tr])
xg = RandomizedSearchCV(xgb.XGBClassifier(tree_method="hist", n_jobs=2, random_state=SEED, verbosity=0),
                        {"max_depth": [2, 3, 4], "learning_rate": [0.03, 0.1], "n_estimators": [200, 400],
                         "colsample_bytree": [0.6, 0.9], "min_child_weight": [1, 5], "subsample": [0.8]},
                        n_iter=8, scoring="roc_auc", cv=cv, n_jobs=1, random_state=SEED)
xg.fit(Xtr, yb.loc[tr], sample_weight=compute_sample_weight("balanced", yb.loc[tr]))

rng = np.random.default_rng(SEED)
yt = yb.loc[te].values
boot = [rng.integers(0, len(yt), len(yt)) for _ in range(1000)]
rows, probs = [], {}
for name, search in [("Logistic regression", lg), ("XGBoost", xg)]:
    p = search.best_estimator_.predict_proba(Xte)[:, 1]; probs[name] = p
    b = [roc_auc_score(yt[i], p[i]) for i in boot]
    rows.append({"Model": name, "CV AUC": round(search.best_score_, 3), "Test AUC": round(roc_auc_score(yt, p), 3),
                 "95% CI low": round(np.percentile(b, 2.5), 3), "95% CI high": round(np.percentile(b, 97.5), 3),
                 "Balanced accuracy": round(balanced_accuracy_score(yt, p > 0.5), 3),
                 "Recall: planners": round(recall_score(yt, p > 0.5), 3), "Best settings": str(search.best_params_)})
intent_scores = pd.DataFrame(rows)
d = np.array([roc_auc_score(yt[i], probs["XGBoost"][i]) - roc_auc_score(yt[i], probs["Logistic regression"][i]) for i in boot])
intent_diff = (roc_auc_score(yt, probs["XGBoost"]) - roc_auc_score(yt, probs["Logistic regression"]), *np.percentile(d, [2.5, 97.5]))
print("XGBoost minus logistic regression, test AUC: %+.4f  (95%% CI %+.4f to %+.4f)" % intent_diff)
intent_scores

## B2. What separates planners from non-planners?

SHAP values from the XGBoost model, on the test firms, grouped into the same 13 themes as Day 3. Positive values push towards "plans to adopt".

In [ ]:
model_b = xg.best_estimator_
contrib = model_b.get_booster().predict(xgb.DMatrix(Xte), pred_contribs=True)[:, :-1]        # test firms x encoded columns

cat_cols = [c for c in PREDICTORS if not pd.api.types.is_numeric_dtype(non[c])]
def original_predictor(encoded):
    block, rest = encoded.split("__", 1)
    if block == "cat": return max((c for c in cat_cols if rest.startswith(c + "_")), key=len)
    return rest.replace("missingindicator_", "")
owner = [original_predictor(n) for n in enc_names]
shap = np.zeros((len(te), len(PREDICTORS)))
for j, p in enumerate(owner):
    shap[:, PREDICTORS.index(p)] += contrib[:, j]

def theme_of(p):
    if p == "country": return "Country"
    if p in ("size", "log_employees"): return "Firm size"
    if p in ("turnover_band", "turnover_trend", "employment_trend"): return "Turnover and growth"
    if p in ("nace_section", "sells"): return "Sector and offer"
    if p in ("log_firm_age", "age_band"): return "Firm age"
    if p == "hiring_difficulty": return "Hiring difficulty"
    if p.startswith("act_") or p in ("n_other_actions", "asked_followups"): return "Other green actions"
    if p == "investment": return "Investment level"
    if p.startswith(("sup_", "ext_")): return "Support used"
    if p.startswith("bar_") or p == "n_barriers": return "Barriers met"
    if p.startswith("help_"): return "Support wanted"
    if p == "climate_strategy": return "Climate strategy"
    return "Green products, jobs, materials"

themes = sorted(set(theme_of(p) for p in PREDICTORS))
shap_theme = np.zeros((len(te), len(themes)))
for j, p in enumerate(PREDICTORS):
    shap_theme[:, themes.index(theme_of(p))] += shap[:, j]
intent_theme = pd.Series(np.abs(shap_theme).mean(axis=0), index=themes).sort_values(ascending=False).round(4)
intent_pred = pd.Series(np.abs(shap).mean(axis=0), index=PREDICTORS).sort_values(ascending=False).round(4)
display(intent_theme.to_frame("Importance (theme)"))
intent_pred.head(12).to_frame("Importance (predictor)")

In [ ]:
test_b = non.loc[te].reset_index(drop=True)

def direction(predictor):
    j = PREDICTORS.index(predictor)
    v = test_b[predictor]
    if pd.api.types.is_numeric_dtype(v) and v.nunique() > 2:
        groups = np.where(v.isna(), "missing", np.where(v <= v.median(), "low half", "high half"))
    else:
        groups = v.astype(str).values
    g = pd.DataFrame({"SHAP": shap[:, j], "plans": yt}).groupby(groups)
    return pd.DataFrame({"Firms": g.size(), "Actually plan %": (100 * g["plans"].mean()).round(1),
                         "Average SHAP": g["SHAP"].mean().round(3)})

# Yes/no predictors: how much does answering "yes" move the prediction, and how do the raw planning rates compare?
rows = []
for p in PREDICTORS:
    v = test_b[p]
    if pd.api.types.is_numeric_dtype(v) and set(v.dropna().unique()) <= {0, 1}:
        j = PREDICTORS.index(p)
        allv = non[p]
        rows.append({"Predictor": p, "Firms answering yes (all non-users)": int(allv.sum()),
                     "Plan % if yes": round(100 * yb[allv == 1].mean(), 1), "Plan % if no": round(100 * yb[allv == 0].mean(), 1),
                     "SHAP effect of yes": round(float(shap[v.values == 1, j].mean() - shap[v.values == 0, j].mean()), 3)})
yes_no_b = pd.DataFrame(rows).set_index("Predictor").sort_values("SHAP effect of yes", key=abs, ascending=False)
print("Yes/no predictors with the largest effect on planning to adopt")
display(yes_no_b.head(12))
print("Barriers only"); display(yes_no_b[yes_no_b.index.str.startswith("bar_")])
for p in ["climate_strategy", "investment", "size"]:
    print(p); display(direction(p))

## B3. Planning rates in the raw data

The model tells us what it relies on. These tables show the plain percentages behind it, for **all** non-users (not only the test firms).

**About barriers.** The barrier questions were only put to firms that take at least one resource-efficiency action. A firm that was never asked is not the same as a firm that was asked and reported no barrier, so the two are kept apart.

In [ ]:
def rate(mask, label):
    return {"Characteristic": label, "Firms": int(mask.sum()), "Plan to adopt %": round(100 * yb[mask].mean(), 4)}
a = non["n_other_actions"]
asked = non["asked_followups"] == 1
rows = [rate(pd.Series(True, index=non.index), "All non-users"),
        rate(a == 0, "Other resource-efficiency actions: none"), rate(a.between(1, 2), "Other resource-efficiency actions: 1 or 2"),
        rate(a.between(3, 4), "Other resource-efficiency actions: 3 or 4"), rate(a.between(5, 6), "Other resource-efficiency actions: 5 or 6"),
        rate(a >= 7, "Other resource-efficiency actions: 7 or 8"),
        rate(non["climate_strategy"] == "Has a strategy", "Climate strategy: has one"), rate(non["climate_strategy"] == "Planning one", "Climate strategy: planning one"),
        rate(non["climate_strategy"] == "No plans", "Climate strategy: no plans"),
        rate(non["investment"] == "0: nothing", "Invests nothing in resource efficiency"), rate(non["investment"] == "2: 1-5%", "Invests 1 to 5% of revenue"),
        rate(non["ext_public_funding"] == 1, "Has used public funding"), rate(non["ext_public_funding"] == 0, "Has not used public funding"),
        rate(asked & (non["n_barriers"] > 0), "Asked about barriers: reports at least one"), rate(asked & (non["n_barriers"] == 0) & (non["bar_none"] == 1), "Asked about barriers: reports none"),
        rate(asked & (non["n_barriers"] == 0) & (non["bar_none"] == 0), "Asked about barriers: does not know"), rate(~asked, "Not asked about barriers (takes no action)"),
        rate(non["size"].str.startswith("Micro"), "Micro (1 to 9 employees)"), rate(non["size"].str.startswith("Small"), "Small (10 to 49)"),
        rate(non["size"].str.startswith("Medium"), "Medium (50 to 249)")]
planning = pd.DataFrame(rows)
extra = pd.DataFrame([rate(a == 8, "All eight other actions"), rate(non["help_grants"] == 1, "Says grants or subsidies would help most"), rate(non["help_grants"] == 0, "Does not say so"),
                      rate(non["climate_strategy"] == "Already climate neutral", "Climate strategy: already climate neutral"), rate(non["climate_strategy"] == "Unknown", "Climate strategy: does not know"),
                      rate(asked, "Asked the follow-up questions")])
print("Investment categories:", non["investment"].nunique())
display(planning.round(1)); extra.round(1)

In [ ]:
# Each barrier separately, among the firms that were asked
bars = [c for c in non.columns if c.startswith("bar_") and c != "bar_none"]
each_barrier = pd.DataFrame([{"Barrier": b, "Firms reporting it": int(non.loc[asked, b].sum()),
                              "Plan % if reported": round(100 * yb[asked & (non[b] == 1)].mean(), 4),
                              "Plan % if not reported (asked firms only)": round(100 * yb[asked & (non[b] == 0)].mean(), 4)} for b in bars])
# Is the gap only because firms that report barriers are the more active ones? Compare within bands of activity.
band = pd.cut(a, [0, 2, 4, 6, 8], labels=["1 or 2", "3 or 4", "5 or 6", "7 or 8"])
within_band = non[asked].assign(band=band[asked], group=np.where(non.loc[asked, "n_barriers"] > 0, "Reports a barrier", "Reports none or does not know")).pivot_table(
    index="band", columns="group", values="plans_renewable", aggfunc=lambda s: round(100 * s.mean(), 4), observed=True)
within_band["Gap (points)"] = within_band["Reports a barrier"] - within_band["Reports none or does not know"]
within_band.index.name = "Other actions taken"
display(each_barrier.round(1)); within_band.round(1)

### Figure 8: what the model uses to tell planners from non-planners

In [ ]:
t = intent_theme.sort_values()
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.barh(t.index, t.values, color=AQUA, height=0.62)
for i, v in enumerate(t.values):
    ax.text(v + t.max() * 0.012, i, f"{v:.3f}", va="center", fontsize=8, color=MUTED)
ax.set_xlim(0, t.max() * 1.12)
ax.set_xlabel("Importance (average absolute SHAP value, test set)")
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.tick_params(length=0)
fig.suptitle("What the model uses to tell planners from non-planners", x=0.02, ha="left", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig("fig8_intention_importance.png", dpi=300); plt.show()

## YOUR TURN

Pick a predictor and see how planning differs across its categories. `Actually plan %` is the raw data; `Average SHAP` is the model's view after accounting for everything else. When the two disagree, another characteristic is doing the work.

Names to try: `"country"`, `"turnover_band"`, `"green_products"`, `"hiring_difficulty"`, `"nace_section"`, `"n_barriers"`.

In [ ]:
LOOK_AT = "country"        # <-- change this and re-run
direction(LOOK_AT).sort_values("Average SHAP", ascending=False)

## Save the outputs

In [ ]:
bar_shap = yes_no_b[yes_no_b.index.str.startswith("bar_") & (yes_no_b.index != "bar_none")]["SHAP effect of yes"]
key4 = pd.DataFrame({"Item": ["Countries", "Raw vs adjusted, purchased (rho)", "Raw vs adjusted, on-site (rho)", "Non-users", "Planners", "Plan % unweighted", "Plan % weighted",
                              "Intention training firms", "Intention test firms", "XGBoost minus logistic, intention AUC", "difference CI low", "difference CI high",
                              "Single-barrier SHAP effect, lowest", "Single-barrier SHAP effect, highest", "Investment categories"],
                     "Value": [len(country), spearmanr(country["Purchased"], country["Purchased (adjusted)"])[0], spearmanr(country["On-site"], country["On-site (adjusted)"])[0],
                               len(non), int(yb.sum()), 100 * yb.mean(), plan_weighted, len(tr), len(te), *intent_diff, bar_shap.min(), bar_shap.max(), non["investment"].nunique()]})
with pd.ExcelWriter("Day4_results.xlsx") as xl:
    key4.round(6).to_excel(xl, sheet_name="Key numbers", index=False)
    holm.round(6).to_excel(xl, sheet_name="Holm and leave-one-out", index=False)
    pd.DataFrame({"Model": list(probs), "Test AUC": [roc_auc_score(yt, probs[m_]) for m_ in probs],
                  "95% CI low": [np.percentile([roc_auc_score(yt[i], probs[m_][i]) for i in boot], 2.5) for m_ in probs],
                  "95% CI high": [np.percentile([roc_auc_score(yt[i], probs[m_][i]) for i in boot], 97.5) for m_ in probs]}).round(6).to_excel(xl, sheet_name="Intention AUC exact", index=False)
    pd.Series(np.abs(shap_theme).mean(axis=0), index=themes, name="Importance").sort_values(ascending=False).round(6).to_excel(xl, sheet_name="Intention themes exact")
    pd.DataFrame({"Predictors": captured["Predictors"], "Test AUC": captured_auc, "Share of the country gain captured %": captured_share}).round(6).to_excel(xl, sheet_name="Country gain exact", index=False)
    planning.to_excel(xl, sheet_name="Planning rates", index=False)
    extra.to_excel(xl, sheet_name="Planning rates extra", index=False)
    each_barrier.to_excel(xl, sheet_name="Each barrier", index=False)
    within_band.to_excel(xl, sheet_name="Barriers within bands")
    for p_ in ["climate_strategy", "investment", "size"]:
        direction(p_).to_excel(xl, sheet_name=f"Dir {p_}"[:31])
    country.sort_values("Purchased", ascending=False).to_excel(xl, sheet_name="Country table")
    corr_wide.to_excel(xl, sheet_name="Country correlations")
    p_wide.to_excel(xl, sheet_name="Correlation p-values")
    captured.to_excel(xl, sheet_name="Country gain captured", index=False)
    intent_scores.to_excel(xl, sheet_name="Intention model scores", index=False)
    intent_theme.to_frame("Importance").to_excel(xl, sheet_name="Intention themes")
    intent_pred.to_frame("Importance").to_excel(xl, sheet_name="Intention predictors")
    yes_no_b.to_excel(xl, sheet_name="Intention yes-no effects")
    direction("country").sort_values("Average SHAP", ascending=False).to_excel(xl, sheet_name="Intention by country")
print("Saved.")
try:
    from google.colab import files
    for f in ["Day4_results.xlsx", "fig7_country_indicators.png", "fig8_intention_importance.png"]:
        files.download(f)
except ImportError:
    pass